# USGS Data Analysis and Comparison

This notebook analyzes new USGS stream and groundwater data and compares it with existing Santa Barbara County sites.
It processes two text files: stream sites and groundwater sites.


## Imports and setup

In [ ]:
# Standard imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import re

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully")

## Load USGS data files

In [ ]:
# ==== Load USGS Stream and Groundwater Data ====
print("=== LOADING USGS DATA FILES ===")

# Paths to your uploaded text files (update these paths as needed)
stream_file = "C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\sb_county_usgs\sb_streams.txt"  # Update with actual filename
groundwater_file = "C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\sb_county_usgs\entire_usa_2022.txt"  # Update with actual filename

# Function to safely load text files with multiple encoding attempts
def load_text_file(filepath, delimiter='\t'):
    """Load text file with fallback encoding options"""
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(filepath, delimiter=delimiter, encoding=encoding, low_memory=False)
            print(f"Successfully loaded {filepath} with {encoding} encoding")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error loading {filepath} with {encoding}: {e}")
            continue
    
    raise ValueError(f"Could not load {filepath} with any encoding")

# Load the files (uncomment when you have the actual file paths)
# stream_data = load_text_file(stream_file)
# groundwater_data = load_text_file(groundwater_file)

# For now, create placeholder variables
stream_data = None
groundwater_data = None

print("\n*** UPDATE THE FILE PATHS ABOVE WITH YOUR ACTUAL FILENAMES ***")
print("Then uncomment the loading lines and run again.")

# Once loaded, display basic info
if stream_data is not None:
    print(f"\nStream data shape: {stream_data.shape}")
    print("Stream data columns:")
    print(stream_data.columns.tolist())
    display(stream_data.head(3))

if groundwater_data is not None:
    print(f"\nGroundwater data shape: {groundwater_data.shape}")
    print("Groundwater data columns:")
    print(groundwater_data.columns.tolist())
    display(groundwater_data.head(3))

## Data exploration and coordinate identification

In [ ]:
# ==== Explore USGS Data Structure ====
print("=== USGS DATA EXPLORATION ===")

def explore_dataframe(df, name):
    """Explore dataframe structure and identify potential coordinate columns"""
    if df is None:
        print(f"{name}: No data loaded yet")
        return
    
    print(f"\n{name.upper()} DATA:")
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
    
    # Look for coordinate columns
    coord_patterns = {
        'latitude': r'(?i)lat|latitude|y_coord|northing|lat_dd',
        'longitude': r'(?i)lon|lng|longitude|x_coord|easting|long_dd|lon_dd'
    }
    
    potential_coords = {}
    for coord_type, pattern in coord_patterns.items():
        matches = [col for col in df.columns if re.search(pattern, str(col))]
        if matches:
            potential_coords[coord_type] = matches
    
    if potential_coords:
        print(f"Potential coordinate columns:")
        for coord_type, cols in potential_coords.items():
            print(f"  {coord_type}: {cols}")
    else:
        print("No obvious coordinate columns found - manual inspection needed")
    
    # Show data types
    print(f"\nData types:")
    for col in df.columns[:10]:  # First 10 columns
        dtype = df[col].dtype
        non_null = df[col].notna().sum()
        print(f"  {col}: {dtype} ({non_null:,}/{len(df):,} non-null)")
    
    if len(df.columns) > 10:
        print(f"  ... and {len(df.columns) - 10} more columns")
    
    # Show sample data
    print(f"\nFirst few rows:")
    return df.head()

# Explore both datasets
if stream_data is not None or groundwater_data is not None:
    stream_sample = explore_dataframe(stream_data, "stream")
    if stream_sample is not None:
        display(stream_sample)
    
    groundwater_sample = explore_dataframe(groundwater_data, "groundwater")
    if groundwater_sample is not None:
        display(groundwater_sample)
else:
    print("Load the data files first by updating the file paths in the previous cell")

## Coordinate processing and standardization

In [ ]:
# ==== Process and Standardize Coordinates ====
print("=== COORDINATE PROCESSING ===")

def process_usgs_coordinates(df, name, lat_col=None, lon_col=None):
    """Process coordinates in USGS data"""
    if df is None:
        print(f"{name}: No data to process")
        return None
    
    print(f"\nProcessing {name} coordinates...")
    
    # If columns not specified, try to auto-detect
    if lat_col is None or lon_col is None:
        print("Coordinate columns not specified - attempting auto-detection")
        
        # Common USGS column patterns
        lat_patterns = [r'(?i)dec_lat', r'(?i)latitude', r'(?i)lat_dd', r'(?i)lat']
        lon_patterns = [r'(?i)dec_long', r'(?i)longitude', r'(?i)long_dd', r'(?i)lon_dd', r'(?i)lon']
        
        for pattern in lat_patterns:
            matches = [col for col in df.columns if re.search(pattern, str(col))]
            if matches:
                lat_col = matches[0]
                break
        
        for pattern in lon_patterns:
            matches = [col for col in df.columns if re.search(pattern, str(col))]
            if matches:
                lon_col = matches[0]
                break
        
        if lat_col and lon_col:
            print(f"Auto-detected coordinates: lat='{lat_col}', lon='{lon_col}'")
        else:
            print("Could not auto-detect coordinate columns. Please specify manually.")
            return None
    
    # Create working copy
    df_processed = df.copy()
    
    # Convert coordinates to numeric
    df_processed['lat_combined'] = pd.to_numeric(df_processed[lat_col], errors='coerce')
    df_processed['lon_combined'] = pd.to_numeric(df_processed[lon_col], errors='coerce')
    
    # Check for coordinate issues
    initial_count = len(df_processed)
    valid_coords = df_processed['lat_combined'].between(-90, 90) & df_processed['lon_combined'].between(-180, 180)
    
    print(f"Coordinate validation:")
    print(f"  Initial rows: {initial_count:,}")
    print(f"  Valid coordinates: {valid_coords.sum():,}")
    print(f"  Invalid/missing coordinates: {(~valid_coords).sum():,}")
    
    # Filter to valid coordinates
    df_processed = df_processed[valid_coords].copy()
    
    # Add data source identifier
    df_processed['data_source'] = f'USGS_{name}'
    
    # Show coordinate ranges
    if not df_processed.empty:
        print(f"Coordinate ranges:")
        print(f"  Latitude: {df_processed['lat_combined'].min():.6f} to {df_processed['lat_combined'].max():.6f}")
        print(f"  Longitude: {df_processed['lon_combined'].min():.6f} to {df_processed['lon_combined'].max():.6f}")
    
    return df_processed

# Process both datasets (update column names as needed after data exploration)
# PLACEHOLDER - UPDATE THESE COLUMN NAMES BASED ON YOUR ACTUAL DATA
STREAM_LAT_COL = None  # e.g., 'dec_lat_va' or 'latitude'
STREAM_LON_COL = None  # e.g., 'dec_long_va' or 'longitude'
GROUNDWATER_LAT_COL = None  # e.g., 'dec_lat_va' or 'latitude'
GROUNDWATER_LON_COL = None  # e.g., 'dec_long_va' or 'longitude'

# Process the data
stream_processed = process_usgs_coordinates(stream_data, 'stream', STREAM_LAT_COL, STREAM_LON_COL)
groundwater_processed = process_usgs_coordinates(groundwater_data, 'groundwater', GROUNDWATER_LAT_COL, GROUNDWATER_LON_COL)

print("\n*** UPDATE THE COORDINATE COLUMN NAMES ABOVE BASED ON YOUR DATA EXPLORATION ***")

## Create GeoDataFrames and filter to study area

In [ ]:
# ==== Create GeoDataFrames and Filter to Study Area ====
print("=== CREATING GEODATAFRAMES ===")

def create_usgs_geodataframe(df_processed, name):
    """Create GeoDataFrame from processed USGS data"""
    if df_processed is None or df_processed.empty:
        print(f"{name}: No processed data available")
        return None
    
    print(f"Creating {name} GeoDataFrame...")
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df_processed,
        geometry=gpd.points_from_xy(df_processed['lon_combined'], df_processed['lat_combined']),
        crs="EPSG:4326"
    )
    
    print(f"  Created GeoDataFrame with {len(gdf):,} points")
    
    return gdf

# Create GeoDataFrames
gdf_stream = create_usgs_geodataframe(stream_processed, 'stream')
gdf_groundwater = create_usgs_geodataframe(groundwater_processed, 'groundwater')

# Filter to Santa Barbara County area (reuse county boundary from previous work)
print("\n=== FILTERING TO SANTA BARBARA COUNTY ===")

# Load Santa Barbara County boundary
sb_county_shapefile = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\sb_county_shapefile\sb_county_shp.shp"

try:
    sb_county = gpd.read_file(sb_county_shapefile)
    if sb_county.crs is None or sb_county.crs.to_string().upper() not in ("EPSG:4326", "WGS84"):
        sb_county = sb_county.to_crs("EPSG:4326")
    
    sb_county_geom = sb_county.union_all() if hasattr(sb_county, "union_all") else sb_county.unary_union
    SB_COUNTY_GDF = gpd.GeoSeries([sb_county_geom], crs="EPSG:4326")
    
    print(f"Loaded Santa Barbara County boundary")
    
    # Filter each dataset to Santa Barbara County
    def filter_to_sb_county(gdf, name):
        if gdf is None:
            return None
        
        within_sb = gdf.within(sb_county_geom)
        gdf_sb = gdf.loc[within_sb].copy()
        
        print(f"{name.capitalize()} sites:")
        print(f"  Total: {len(gdf):,}")
        print(f"  In Santa Barbara County: {len(gdf_sb):,}")
        print(f"  Outside county: {(~within_sb).sum():,}")
        
        return gdf_sb
    
    gdf_stream_sb = filter_to_sb_county(gdf_stream, 'stream')
    gdf_groundwater_sb = filter_to_sb_county(gdf_groundwater, 'groundwater')
    
except FileNotFoundError:
    print(f"Santa Barbara County shapefile not found: {sb_county_shapefile}")
    print("Using full datasets without county filtering")
    gdf_stream_sb = gdf_stream
    gdf_groundwater_sb = gdf_groundwater
    SB_COUNTY_GDF = None

# Show summary
print("\n=== USGS DATA SUMMARY ===")
if gdf_stream_sb is not None:
    print(f"Stream sites in study area: {len(gdf_stream_sb):,}")
if gdf_groundwater_sb is not None:
    print(f"Groundwater sites in study area: {len(gdf_groundwater_sb):,}")

total_usgs = 0
if gdf_stream_sb is not None:
    total_usgs += len(gdf_stream_sb)
if gdf_groundwater_sb is not None:
    total_usgs += len(gdf_groundwater_sb)

print(f"Total USGS sites in study area: {total_usgs:,}")

## Load existing Santa Barbara data for comparison

In [ ]:
# ==== Load Existing Santa Barbara County Data for Comparison ====
print("=== LOADING EXISTING SB COUNTY DATA ===")

# This assumes you have the CSV export from your previous Santa Barbara analysis
# Update this path to match where you saved the SB county data
sb_existing_csv = "plots/[your_run_id]/santa_barbara_county/santa_barbara_sites.csv"

try:
    # Try to load existing Santa Barbara data
    df_sb_existing = pd.read_csv(sb_existing_csv)
    
    # Create GeoDataFrame
    gdf_sb_existing = gpd.GeoDataFrame(
        df_sb_existing,
        geometry=gpd.points_from_xy(df_sb_existing['lon_combined'], df_sb_existing['lat_combined']),
        crs="EPSG:4326"
    )
    
    print(f"Loaded existing Santa Barbara data: {len(gdf_sb_existing):,} sites")
    
    # Add data source identifier
    gdf_sb_existing['data_source'] = 'Historical_USGS'
    
    print("Water type distribution in existing data:")
    if 'water_type_clean' in gdf_sb_existing.columns:
        print(gdf_sb_existing['water_type_clean'].value_counts())
    
except FileNotFoundError:
    print(f"Could not find existing SB data at: {sb_existing_csv}")
    print("You can either:")
    print("1. Update the path above to your actual SB county CSV file")
    print("2. Or rerun the Santa Barbara analysis to generate the CSV")
    print("3. Or continue with just the new USGS data")
    gdf_sb_existing = None

# Create comparison summary
print("\n=== DATA COMPARISON SUMMARY ===")
print(f"New USGS stream sites: {len(gdf_stream_sb) if gdf_stream_sb is not None else 0:,}")
print(f"New USGS groundwater sites: {len(gdf_groundwater_sb) if gdf_groundwater_sb is not None else 0:,}")
print(f"Existing historical sites: {len(gdf_sb_existing) if gdf_sb_existing is not None else 0:,}")
print(f"Total sites for analysis: {(len(gdf_stream_sb) if gdf_stream_sb is not None else 0) + (len(gdf_groundwater_sb) if gdf_groundwater_sb is not None else 0) + (len(gdf_sb_existing) if gdf_sb_existing is not None else 0):,}")

## Ready for analysis and mapping!

You now have:
- `gdf_stream_sb`: New USGS stream sites in Santa Barbara County
- `gdf_groundwater_sb`: New USGS groundwater sites in Santa Barbara County  
- `gdf_sb_existing`: Your existing historical water sites (if loaded)
- `SB_COUNTY_GDF`: Santa Barbara County boundary for mapping

Next steps:
1. **Update file paths** in the data loading cell with your actual filenames
2. **Update coordinate column names** after exploring your data structure
3. **Add mapping and comparison cells** below
4. **Create visualizations** comparing the datasets
